In [ ]:
import pandas as pd
import requests
import os

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


sep = "|"
decimal = "."
encoding = "UTF-8-sig"
path = r"../data/raw/WeatherData_raw.csv"

if os.path.isdir(r"../images/weather_condition_icons") is False:
    os.mkdir(r"../images/weather_condition_icons")


raw_df = pd.read_csv(filepath_or_buffer=path, decimal=decimal, encoding=encoding, sep=sep)

condition_icons_url_list = raw_df["condition_icon"].unique().tolist()
condition_text_list = raw_df["condition_text"].unique().tolist()
condition_images_path_list = []


for index in range(0, len(condition_icons_url_list)):
    url = condition_icons_url_list[index]
    response = requests.get(f"http:{url}")
    img_data = response.content

    file = open(f"../images/weather_condition_icons/{index}.png", "wb")
    file.write(img_data)
    file.close()

    condition_images_path_list.append(f"../images/weather_condition_icons/{index}.png")


weather_condition_df = pd.DataFrame(
    {
        "condition_id": range(len(condition_icons_url_list)), 
        "condition_icon_url": condition_icons_url_list, 
        "condition_image_path": condition_images_path_list
    }
)
icon_text_df = raw_df[["condition_icon", "condition_text"]].drop_duplicates()
weather_condition_df = pd.merge(left = weather_condition_df, right = icon_text_df, how = "inner", left_on = "condition_icon_url", right_on = "condition_icon")
weather_condition_df = weather_condition_df[["condition_id", "condition_icon_url", "condition_image_path", "condition_text"]]

merge_df = pd.merge(left = raw_df, right = weather_condition_df, left_on = "condition_icon", right_on = "condition_icon_url")


df = merge_df[[
    "county_id", "county", "city", "county_abbrev", "data_type", "time", "is_day", "temp_c", "feelslike_c", "heatindex_c", "cloud",
    "condition_id", "humidity", "wind_kph", "gust_kph", "wind_degree", 
    "wind_dir", "windchill_c", "pressure_mb", "precip_mm", 
    "will_it_rain", "chance_of_rain", "will_it_snow", "snow_cm",
    "chance_of_snow", "vis_km", "uv"
]]

df = df.rename(columns = {"time": "date_time", "temp_c": "temp", "gust_kph": "wind_gust_kph", "vis_km": "visibility_km"})

df["date_time"] = pd.to_datetime(df["date_time"])


if os.path.isdir(r'../data/processed') is False:
    os.mkdir(r'../data/processed')

with pd.ExcelWriter(r'../data/processed/WeatherData_processed.xlsx') as writer:  
    df.to_excel(writer, sheet_name='Weather data', header = True, index = False)
    weather_condition_df.to_excel(writer, sheet_name='weather_condition', header = True, index = False)




print("The data has been processed!")
count = df.shape[0]
colums_count = df.shape[1]
print(f"Rows count: {count}\nColums count: {colums_count}")

df.head(30)